# SEC analysis (Figure 4)

This notebook builds on **SAPP_DMX** by **Basile Wicky** (Institute for Protein
Design) for SEC chromatogram parsing, calibration and pooling. The SEC engine
(`wetlab_utils`, `SEC_pool_and_norm`) is **not** redistributed here — it is
obtained from that repository:

> B. Wicky et al., *SAPP_DMX*. https://github.com/bwicky/SAPP_DMX

Please cite SAPP_DMX if you use this notebook. Setup and the external inputs
required for full reproduction are documented in `REPRODUCE_figure4_SEC.md`.

In [ ]:
# =====================================================================
# CONFIG - portable paths + external inputs for full SEC reproduction
# ---------------------------------------------------------------------
# This notebook reproduces Figure 4 (library-scale SEC yields). Its SEC
# engine is NOT bundled: it lives in Basile Wicky's SAPP_DMX repository,
# and the raw HPLC traces + column calibration are large instrument data
# held outside this deposit. See code/REPRODUCE_figure4_SEC.md.
#
#   git clone https://github.com/bwicky/SAPP_DMX.git
#   pip install pycorn
# =====================================================================
import os, sys
from pathlib import Path

_cwd = Path.cwd()
REPO = next((p for p in [_cwd, *_cwd.parents]
             if (p / 'construct').is_dir() and (p / 'examples').is_dir()), _cwd)
OUTDIR = REPO / 'code' / 'outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTDIR)                       # figures/tables are written here
Path('SEC_outputs').mkdir(exist_ok=True)

# SAPP_DMX provides wetlab_utils (parse_chromatograms, process_sec_data,
# yields, calibrated_results, protparam, select_fractions, gen_ot2_script)
SAPP_DMX_DIR = Path(os.environ.get('SAPP_DMX_DIR', Path.home() / 'SAPP_DMX'))
sys.path.append(str(SAPP_DMX_DIR))

# --- Inputs ----------------------------------------------------------
# Well -> design map. 2_cloning_qc.ipynb writes OUTDIR/sec_input.csv for the
# synthetic demo; for the paper's Figure 4 point this at your 60-variant file.
SEC_INPUT = Path(os.environ.get('SEC_INPUT', OUTDIR / 'sec_input.csv'))
# Column calibration (.json) and raw HPLC trace folders (glob) - NOT shipped:
SEC_CALIBRATION = os.environ.get('SEC_CALIBRATION', 'REPLACE_ME/231123_S75_5-150_HPLC.json')
SEC_TRACES_GLOB = os.environ.get('SEC_TRACES_GLOB', 'REPLACE_ME/20240517_dsPep/*')
SEC_POOL_TEMPLATE = str(SAPP_DMX_DIR / 'SEC_pool_and_norm_v3.py')
print('Deposit root:', REPO, '| SAPP_DMX:', SAPP_DMX_DIR)

# Cowboy results for X

***Add notes about your experiment here.***

*NB* requires a kernel with pycorn.

In [ ]:
import pycorn
import openpyxl

In [ ]:
import glob
from datetime import date; today=date.today().strftime('%Y%m%d')[2:]; print(f'Today: {today}')
# import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from scipy.cluster import hierarchy
import sys, os  # (SAPP_DMX path added in the CONFIG cell above)
import datetime
from wetlab_utils import *
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns; sns.set_theme(
    context='talk', 
    palette='colorblind',
    style='ticks',
    rc={'axes.spines.right':False, 'axes.spines.top':False}
)

import json
import fastcluster
import datetime


## ECHO cloning


`eblocks_plates`: dictionary containing paths to `.xlsx` files from IDT eBlocks order(s).
- If the IDT order spreadsheet is passed unmodified, and a set of vectors is indicated, then the ECHO transfer protocol will include cloning of every eBlock into every vector.
- If more complex cloning combinatorics is desired, include a `Vector` column in the spreadhseet to specify the entry vector for each eBlock. 

***If not using the ECHO generator function, `echo_df` should at a minimum contain the following columns:***
   - `Name`: the name of your design.
   - `Destination Well`: the destination well of the transfer.
   - `Destination Plate Name`: 96w expression plate identifier.
   - `eblock`: the eBlock sequence.
   - `Vector`: the GGA entry vector ID, e.g. `LM0627`.

## Add to create the echo_df with columns needed
 >Source plate: Number of plate - ensure they have the same number in the HPLC and the PCR/Cloning plate
 >Destination Well -> source well:  as the protocol works in 96 well format this will always be like this unless it is changed
 >Destination Plate Name > should be the same as source plate
 >96w --> well > same as source and destination - don't know yet if it is essential

In [ ]:
echo_df = pd.read_csv(SEC_INPUT)
echo_df['96w'] = echo_df['Well']
echo_df['Source Well'] = echo_df['Well']
echo_df = echo_df.rename(columns={"name": "Name",  "Well": "Destination Well", "Source Plate Name":"Destination Plate Name"})
echo_df['Source Plate Name'] = echo_df['Destination Plate Name']
echo_df['Vector'] = '1_yfb001_periplasmic_peptide_expression_pet-24b'

In [ ]:
## Only here because in this case there were over 96 samples and would just not work here
echo_df = echo_df.drop(echo_df.index[96:120])

## Expression/purification parameters
- `oligomers`: whether or not your designs are oligomers (if `True`, a column `n_chains` needs to be specified in `echo_df`).
- `culture_vol`: culture volume, formated as a string `AxB` where `B` is the volume of one culture (in mL), and `A` the number of consolidated cultures.
- `culture_time`: time between inocculation and harvest, in hours.
- `culture_temp`: temperature during growth/expression, in degree Celsius.
- `sec_column`: column used for SEC, e.g. `S75_5-150`.
- `sec_buffer`: buffer used for SEC.
- `sec_buffer_pH`: the pH of the SEC buffer. 
- `sec_instrument`: the instrument used for SEC. Must be either `akta` or `hplc`.
- `sec_calibration`: instrument and column-specific calibration file (`.json` format)
- `sec_traces`: dictionary of lists of paths to exported SEC traces. 
    - `AKTA` data structure should be `/path_to_your_run/*.zip`. The list should contain paths to the individual `.zip` files.
    - `HPLC` data structure should be `/path_to_your_run/trace_folders/*.CSV`. The list should contain paths to the individual *trace folders*, NOT the individual `.CSV` files. The folder containing the individual trace folders should also contain the `fractions.csv` file containing the mapping between traces and fractions.

In [ ]:
oligomers       = False
culture_vol     = '4x1'
culture_time    = 16
culture_temp    = 25
sec_column      = 'S75_5-150'
sec_buffer      = 'TBS'
sec_buffer_pH   = 7.4
sec_instrument  = 'hplc'
sec_calibration = SEC_CALIBRATION
sec_traces      = {
    1:glob.glob(SEC_TRACES_GLOB)
}

print(f'Number of SEC traces: {np.sum([len(v) for v in sec_traces.values()])}')
print('\n'.join([f'> {k}: {len(v)}' for k, v in sec_traces.items()]))

### Add information to dataframe

If you are cloning into non-standard entry vectors, change the path to their `.fa` files.

In [ ]:
# # Clone.
# #echo_df['plasmid_seq'] = echo_df.apply(
#     clone, 
#     path_to_vectors='/software/lab/johnbercow/entry_vectors/',
#     axis=1,
#)
#echo_df['ORF'] = echo_df.apply(coding_seq, axis=1)
echo_df['ORF'] = echo_df['ORF']

# Protparam.
echo_df = protparam(echo_df, oligomers, sec_buffer_pH)

# Expression parameters.
echo_df['culture_vol'] = float(culture_vol.split('x')[0]) * float(culture_vol.split('x')[1])
echo_df['culture_time'] = culture_time
echo_df['culture_temp'] = culture_temp

# Purification parameters.
echo_df['sec_instrument'] = sec_instrument
echo_df['sec_column'] = sec_column
echo_df['sec_buffer'] = sec_buffer  

# OPTIONAL: define a category for later comparisons (e.g. designs made using different methods).
echo_df['category'] = echo_df['Destination Plate Name'].astype(str)
echo_df.category.value_counts()

### Process SEC chromatograms

*NB*: 205 nm may be used instead of 280 nm for protein quantification, however this option is poorly validated.

In [ ]:
# %%capture

wl = 280 # wavelength used for all downstream data processing.

parsed_chromatograms = {}
for run, traces in sec_traces.items():
    parsed_chromatograms[run] = parse_chromatograms(traces, instrument=sec_instrument)
    
sec_data = {}
for run, parsed_chroma in parsed_chromatograms.items():
    sec_data[run] = process_sec_data(parsed_chroma, wavelength=wl)

### Map traces to wells

***CAUTION:*** make sure to validate your mapping (both wells AND destination plates), especially if you had to e.g. restart you SEC run (indices may have shifted!). 

Check that the size of the resulting dataframe matches the number of samples you injected.

In [ ]:
sec_df = pd.DataFrame()
for run, sec_d in sec_data.items():
    
    w96 = [str(p) + '_' + r + str(c) for p in range(1, 10) for r in 'ABCDEFGH' for c in range(1, 13)]

    if sec_instrument == 'akta':
        
        akta2well = {i+1:w for i, w in enumerate(w96)}    
        sec_d['well_id'] = sec_d['trace_id'].map(akta2well)
        sec_d['Destination Plate Name'] = sec_d['well_id'].apply(lambda x: int(x.split('_')[0]))
        sec_d['Destination Well'] = sec_d['well_id'].apply(lambda x: x.split('_')[1])
        
    if sec_instrument == 'hplc':
        
        drawer2plate = {'D1F':1, 'D1B':2, 'D2F':3, 'D2B':4, 'D3F':5, 'D3B':6, 'D4F':7, 'D4B':8}
        sec_d['Destination Plate Name'] = sec_d['trace_id'].apply(lambda x: drawer2plate[x.split('-')[0]])
        sec_d['Destination Well'] = sec_d['trace_id'].apply(lambda x: x.split('-')[1])
        sec_d['well_id'] = sec_d['Destination Plate Name'].astype(str) + '_' + sec_d['Destination Well']
                                                                   
    sec_df = pd.concat([sec_df, sec_d])
    
echo_df['Destination Plate Name'] = echo_df['Destination Plate Name'].astype(str)
sec_df['Destination Plate Name'] = sec_df['Destination Plate Name'].astype(str)
echo_df['Destination Well'] = echo_df['Destination Well'].astype(str)
sec_df['Destination Well'] = sec_df['Destination Well'].astype(str)
    
df = echo_df.merge(
    sec_df, 
    left_on=['Destination Plate Name', 'Destination Well'], 
    right_on=['Destination Plate Name', 'Destination Well'],
).reset_index(drop=True)

len(df)

### Add yields and calibrated results to dataframe

In [ ]:
df = yields(df) 
df, CI95, CI99 = calibrated_results(df, sec_calibration) 
int_df = df.copy()

## Results overview

In [ ]:
sns.boxplot(
    data=df, 
    x='category', 
    y='tot_yield', 
    fliersize=0,
    color='white',
)

sns.swarmplot(
    data=df, 
    x='category', 
    y='tot_yield', 
    alpha=0.8, 
    edgecolor='k', 
    linewidth=1, 
    size=6,
)
plt.ylabel('Tot. yield / mg')
plt.tight_layout()
plt.savefig('SEC_outputs/yields_by_category.png', dpi=300)
plt.show()

In [ ]:
for cat, sel in df.groupby('category'):

    sel.reset_index(inplace=True)

    # Cluster chromatograms. 
    vol_light = np.vstack(sel.loc[:, 'vol_light'].to_numpy())
    Abs_light = np.vstack(sel.loc[:, 'Abs_light'].to_numpy())
    clustered_idx = hierarchy.dendrogram(
            hierarchy.linkage(
            Abs_light, 
            method='average',
            optimal_ordering=True
        ), 
        no_plot=True,
    )['leaves'] # 'leaves' are the re-ordered indices
    
    # Plot.
    delta = np.max([np.max(x) for x in sel.Abs.to_numpy()]) / 20
    fig, ax = plt.subplots(ncols=2, figsize=(10, 5))
    for i, r in sel.iterrows():
        ax[0].plot(r.vol, r.Abs, color='C0', alpha=0.1)
        ax[1].fill_between(
            x=vol_light[clustered_idx[i]],
            y1=Abs_light[clustered_idx[i]] + i*delta,
            y2=i*delta, 
            color='C0',
            alpha=0.1, 
            zorder=i
        )

    ax[0].set(xlabel='Retention vol. / mL', ylabel=f'A280 / mAU')
    ax[1].set(xlabel='Retention vol. / mL', yticks=[])
    ax[1].spines['left'].set_visible(False)
    ax[0].set(title=f'{cat} ($N = {len(sel)}$)')
    plt.tight_layout()
    filename = f'all_sec_traces_{cat}.png'
    subfolder = 'SEC_outputs/'
    filepath = os.path.join(subfolder, filename)

    plt.savefig(filepath, dpi=300)
    plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10, 4))

sns.histplot(
    data=df, 
    x='tot_yield', 
    stat='count',
    ax=ax[0],
    log_scale=True, 
    element='step', 
    alpha=0.1,
)

ax0 = ax[0].twinx()
ax0.vlines(df.tot_yield.median(), 0, .5, color='salmon')
ax0.scatter(df.tot_yield.median(), .5, color='salmon', zorder=10)
sns.ecdfplot(data=df, x='tot_yield', alpha=0.7, ax=ax0)
ax[0].spines['right'].set_visible(True)
ax[0].set_xlabel('Total soluble yield / mg')
ax[0].set_title(f'Median = {df["tot_yield"].median():.2f} mg\n($N$ = {len(df)})')
          
sns.histplot(
    data=df, 
    x='tot_yield', 
    hue='category',
    stat='count',
    ax=ax[1],
    log_scale=True,
    element='step',
    alpha=0.1,
    legend=False,
)

sns.ecdfplot(
    data=df, 
    x='tot_yield', 
    hue='category', 
    ax=ax[1].twinx(), 
    legend=True,
    alpha=0.7,
)

ax[1].spines['right'].set_visible(True)
ax[1].set_xlabel('Total soluble yield / mg')

plt.tight_layout()
plt.savefig('SEC_outputs/tot_soluble_yields.png', dpi=300)
plt.show()

## Calibrated results

In [ ]:
##### ERROR ON CALIBRATION CORRECTED #####
# Load SEC calibration parameters
with open(sec_calibration, 'r') as f:
    sec_cal = json.load(f)

Vo, Vc = sec_cal['Vo'], sec_cal['Vc']
intercept, slope = sec_cal['intercept'], sec_cal['slope']
all_mws = np.hstack([sec_cal['log10mw'], np.log10(df['MW'].to_numpy())])
xs = np.linspace(all_mws.min() - 0.2, all_mws.max() + 0.2, 100)

def Vel2MW(Vel):
    return 10**((((Vel - Vo) / (Vc - Vo)) - intercept) / slope)

# Iterate over categories in the dataframe
for c, sel in df.groupby('category'):
    plt.figure(figsize=(10, 6))
    
    # Plot calibration curve and CI
    plt.scatter(
        [10**x for x in sec_cal['log10mw']], 
        sec_cal['Kav'], 
        edgecolor='grey', 
        linewidth=1, 
        facecolor='none', 
        alpha=1,
    )
    
    plt.plot(
        10**xs, 
        intercept + slope * xs, 
        color='grey', 
        linewidth=1,
    )
    
    # Placeholder for CI95 values (update this with actual CI95 values if available)
    CI95 = [intercept + slope * xs - 0.1, intercept + slope * xs + 0.1]
    plt.fill_between(
        10**xs, 
        CI95[0], 
        CI95[1], 
        color='grey', 
        alpha=0.2,
        edgecolor=None,
    )

    # Plot data with color mapping
    norm = plt.Normalize(sel.tot_yield.min(), sel.tot_yield.max())
    sm = plt.cm.ScalarMappable(cmap='Spectral_r', norm=norm)
    sm.set_array([])  # Needed for the colorbar
    
    ax = sns.scatterplot(
        data=sel, 
        x='MW', 
        y='main_peak_norm_retention', 
        hue='tot_yield', 
        palette='Spectral_r', 
        zorder=10, 
        edgecolor='k', 
        linewidth=1, 
        alpha=0.7,
    )
    
    plt.xscale('log')
    plt.xlabel('MW / Da')
    plt.ylabel('Norm. retention / a.u.')
    
    # Add colorbar
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Tot. yield / mg')
    
    ax.get_legend().remove()
    plt.title(c)
    plt.tight_layout()
    
    plot = f'{c}_against_calibration.png'
    subdir = 'SEC_outputs/'
    filepath = os.path.join(subdir, plot)
    plt.savefig(filepath, dpi=300)
    plt.show()


In [ ]:
cat2col = {cat: 'C' + str(i) for i, cat in enumerate(df.category.unique())}

# Create the clustermap
g = sns.clustermap(
    np.array(df['Abs_norm_light'].to_list()),  # Ensure this is a 2D array
    method='average',
    cmap='bone_r',
    col_cluster=False,
    row_colors=[cat2col[cat] for cat in df.category.to_list()],
    tree_kws={'lw': 1, 'color': 'grey'},
    row_linkage=fastcluster.linkage(np.array(df['Abs_norm_light'].to_list()), method='average')  # Use fastcluster
)

g.cax.remove()
plt.yticks([])

# X-axis
nlabels = 7
vmin, vmax = np.min(df['vol_light'].min()), np.max(df['vol_light'].max())
vol_labels = np.linspace(vmin, vmax, nlabels, dtype=float)
mw_labels = [Vel2MW(v) for v in vol_labels]
lbls = [f'{v:.2f}\n[{mw/1000:.0f}]' for v, mw in zip(vol_labels, mw_labels)]
plt.xticks(ticks=np.linspace(0, 100, nlabels), labels=lbls, rotation=0)

# Legend
if len(cat2col) > 0:  # Ensure there are categories
    handles = [Patch(edgecolor='k', facecolor=v) for v in cat2col.values()]
    plt.legend(
        handles,
        cat2col.keys(),
        title='category',
        ncol=max(1, len(cat2col) // 2),  # Ensure ncol is at least 1
        bbox_to_anchor=(0.55, 0.92),
        bbox_transform=plt.gcf().transFigure,
        loc='upper center',
    )

plt.xlabel('Retention vol. / mL\n[MW / kDa]')
plt.savefig('SEC_outputs/clustered_sec_traces.png', dpi=300)
plt.show()

In [ ]:
# #### original from BASIL BUT GAVE ERRROR ---


# # Get SEC column calibration parameters.
# with open(sec_calibration, 'r') as f:
#     sec_cal = json.load(f)

# Vo, Vc = sec_cal['Vo'], sec_cal['Vc'] 
# intercept, slope = sec_cal['intercept'], sec_cal['slope'] 
# all_mws = np.hstack([sec_cal['log10mw'], np.log10(df['MW'].to_numpy())])
# xs = np.linspace(all_mws.min()-0.2, all_mws.max()+0.2, 100)
    
# def Vel2MW(Vel):
#         return 10**((((Vel - Vo) / (Vc - Vo)) - intercept) / slope)
    
# for c, sel in df.groupby('category'):
    
#     # Plot calibration curve and CI.
#     plt.scatter(
#         [10**x for x in sec_cal['log10mw']], 
#         sec_cal['Kav'], 
#         edgecolor='grey', 
#         linewidth=1, 
#         color='None', 
#         alpha=1,
#     )
    
#     plt.plot(
#         10**xs, 
#         intercept + slope * xs, 
#         color='grey', 
#         linewidth=1,
#     )
    
#     CI95 = [intercept + slope * xs - 0.1, intercept + slope * xs + 0.1]
#     plt.fill_between(
#         10**xs, 
#         CI95[0], 
#         CI95[1], 
#         color='grey', 
#         alpha=0.2, 
#         edgecolor=None,
#     )

#     # Plot data.
#     norm = plt.Normalize(sel.tot_yield.min(), sel.tot_yield.max())
#     sm = plt.cm.ScalarMappable(cmap='Spectral_r', norm=norm)
#     sm.set_array([])
    
#     ax = sns.scatterplot(
#         data=sel, 
#         x='MW', 
#         y='main_peak_norm_retention', 
#         hue='tot_yield', 
#         palette='Spectral_r', 
#         zorder=10, 
#         edgecolor='k', 
#         linewidth=1, 
#         alpha=0.7,
#     )
    
#     plt.xscale('log')
#     plt.xlabel('MW / Da')
#     plt.ylabel('Norm. retention / a.u.')
    
#     cbar = plt.colorbar(sm, ax=ax)
#     cbar.set_label('Tot. yield / mg')
    
#     ax.get_legend().remove()
#     ax.figure.colorbar(sm, label='Tot. yield / mg')
#     plt.title(c)
#     plt.tight_layout()
#     plt.savefig(f'{c}_against_calibration.png', dpi=300)
#     plt.show()

In [ ]:
# Assuming df is your DataFrame and is already loaded

# Normalize and create ScalarMappable for color mapping
norm = plt.Normalize(df.tot_yield.min(), df.tot_yield.max())
sm = plt.cm.ScalarMappable(cmap='Spectral_r', norm=norm)
sm.set_array([])

# Create the boxplot
plt.figure(figsize=(12, 8))
sns.boxplot(
    data=df, 
    x='category', 
    y='expected_norm_retention', 
    zorder=0,
    medianprops={'visible': True, 'color': 'lightgrey', 'ls': '-', 'lw': 10},
    whiskerprops={'visible': False},
    showfliers=False,
    showbox=False,
    showcaps=False,
)

# Create the swarmplot
ax = sns.swarmplot(
    data=df, 
    x='category', 
    y='main_peak_norm_retention', 
    hue='tot_yield', 
    palette='Spectral_r', 
    alpha=0.7, 
    edgecolor='k', 
    linewidth=1, 
    size=6,
)

# Add colorbar
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Tot. yield / mg')

# Customize the plot
ax.get_legend().remove()
ax.set(ylim=[-0.1, 1], xlabel='', ylabel='Norm. retention / a.u.')
plt.tight_layout()

# Save and show the plot
plt.savefig('SEC_outputs/norm_retention_by_category.png', dpi=300)
plt.show()

## Peaks distribution

In [ ]:
xs = np.hstack(df.peaks.to_list())
ys = np.hstack(df.peak_heights.to_list())
g = sns.jointplot(
    x=xs, 
    y=ys, 
    kind='kde', 
    ratio=4,
)
g.ax_joint.scatter(xs, ys, facecolors='none', edgecolor='salmon')
g.ax_joint.set_xlabel('Peak retention volume / mL')
g.ax_joint.set_ylabel('Normalized peak height')
plt.tight_layout()
plt.savefig('SEC_outputs/peaks_distribution.png', dpi=300)
plt.show()

In [ ]:
sns.stripplot(
    data=df.sort_values(by='main_peak_agg_state'),
    x='main_peak_agg_state', 
    y='main_peak_height',
    hue='category', 
    alpha=0.5, 
    edgecolor='k', 
    linewidth=1, 
    size=6,
    dodge=True,
)

plt.xlabel('Aggregation state')
plt.ylabel('Peak height / mAU')
plt.tight_layout()
plt.savefig('SEC_outputs/aggregation_states.png', dpi=300)
plt.show()

## Per-well results

In [ ]:

# Assuming df is the DataFrame and Vel2MW is defined above

# Iterate over the plates
for plate, sel in df.groupby('Destination Plate Name'):
    
    states = [str(i) for i in range(9)] + ['9+']
    
    agg_df = pd.DataFrame(
        index=sel['Destination Well'].to_list(),
        columns=states,
    )
    
    agg_df = agg_df.merge(
        sel[['tot_yield', 'Destination Well']], 
        left_index=True, 
        right_on='Destination Well'
    ).set_index('Destination Well', drop=True)
    
    agg_df.fillna(0, inplace=True)
    
    for i, r in sel.iterrows():
        agg_states = [round(Vel2MW(v) / r.protomer_MW) for v in r.peaks]
        proportions = r.tot_yield * r.peak_heights / r.peak_heights.sum()

        for agg_st, prop in zip(agg_states, proportions):

            if agg_st < 9:
                agg_df.loc[r['Destination Well'], str(agg_st)] += prop

            else:
                agg_df.loc[r['Destination Well'], '9+'] += prop

    agg_df.sort_values(
        by=['tot_yield'] + states,
        ascending=False,
        inplace=True,
    )
   
    n_subplots = np.ceil(len(agg_df) / 48).astype(int)
    fig, ax = plt.subplots(
        nrows=n_subplots, 
        figsize=(len(agg_df) / n_subplots * 0.3, 5 * n_subplots),
        sharey=True,
    )

    if n_subplots == 1:
        ax = [ax]

    colormap = plt.get_cmap('magma')  # Use plt.get_cmap directly

    for i in range(n_subplots):
        agg_df.iloc[48 * i:48 * (i + 1)][states].plot(
            ax=ax[i],
            kind='bar', 
            stacked=True, 
            edgecolor='k',
            alpha=0.80,
            colormap=colormap,  # Using the colormap object directly
        )

        ax[i].legend(bbox_to_anchor=(1, 1), title='Agg. state')
        ax[i].set(xlabel='', ylabel='Total soluble yield / mg')
        ax[i].set_title(f'Plate {plate} ({48 * i + 1}-{48 * (i + 1)})')
    
    plt.tight_layout()
    filename = f'plate_{plate}_agg_per_well.png'
    subfolder = 'SEC_outputs/'
    filepath = os.path.join(subfolder,filename)
    plt.tight_layout()
    plt.savefig(filepath, dpi=300)
    plt.show()


In [ ]:

# ####ORIGINAL CODE FORM BASIL WITH ERROR ABOUT COLOR MAPPING INCOMPATIBILITY####

# for plate, sel in df.groupby('Destination Plate Name'):
    
#     states = [str(i) for i in range(9)] + ['9+']
    
#     agg_df = pd.DataFrame(
#         index=sel['Destination Well'].to_list(),
#         columns=states,
#     )
    
#     agg_df = agg_df.merge(
#         sel[['tot_yield', 'Destination Well']], 
#         left_index=True, 
#         right_on='Destination Well'
#     ).set_index('Destination Well', drop=True)
    
#     agg_df.fillna(0, inplace=True)
    
#     for i, r in sel.iterrows():
#         agg_states = [round(Vel2MW(v) / r.protomer_MW) for v in r.peaks]
#         proportions = r.tot_yield * r.peak_heights / r.peak_heights.sum()

#         for agg_st, prop in zip(agg_states, proportions):

#             if agg_st < 9:
#                 agg_df.loc[r['Destination Well'], str(agg_st)] += prop

#             else:
#                 agg_df.loc[r['Destination Well'], '9+'] += prop

#     agg_df.sort_values(
#         by=['tot_yield'] + states,
#         ascending=False,
#         inplace=True,
#     )
   
#     n_subplots = np.ceil(len(agg_df)/48).astype(int)
#     fig, ax = plt.subplots(
#         nrows=n_subplots, 
#         figsize=(len(agg_df)/n_subplots*.3, 5*n_subplots),
#         sharey=True,
#     )

#     if n_subplots == 1:
#         ax = [ax]

#     for i in range(n_subplots):
#         agg_df.iloc[48*i:48*(i+1)][states].plot(
#             ax=ax[i],
#             kind='bar', 
#             stacked=True, 
#             edgecolor='k',
#             alpha=0.80,
#             cmap='magma',
#         )

#         ax[i].legend(bbox_to_anchor=(1,1), title='Agg. state')
#         ax[i].set(xlabel='', ylabel='Total soluble yield / mg')
#         ax[i].set_title(f'Plate {plate} ({48*i+1}-{48*(i+1)})')
#     plt.tight_layout()
#     plt.savefig(f'plate_{plate}_agg_per_well.png', dpi=300)
#     plt.show()


## Pool fractions

- Choose the number of fraction to pool with `n_fractions`. Only adjacent fractions will be picked.
    - The tolerance (in mL) for what qualifies as adjacent can be specified with `adjacency`.

<!-- -->
- Choose the fraction picking logic with `how`:
    - `largest` will pick the fraction with the largest integral.
    - `nearest` will pick the fraction containing the *peak* nearest to the expected elution volume.

<!-- -->
- Use `manual_edits` if you want to overwrite the fractions that are automatically picked.


In [ ]:
manual_edits = np.array([
    ['well_id', ['fraction1', 'fraction2', ]],
], dtype=object)

In [ ]:
from numpy import AxisError
pooled_df = pd.DataFrame(
    columns=['pooled_fractions', 'pooled_frac_vol', 'pooled_vol', 'pooled_integral', 'pooled_peak_Abs', 'pooled_peak_vol']
)
pooled_df


for w, r in df.set_index('well_id').iterrows():
    try:
        select_fractions(
            w, 
            r, 
            pooled_df, 
            manual_edits, 
            Vel2MW, 
            wl, 
            n_fractions=3,
            how='nearest',
            adjacency=0.025,
        )
        # Add plt.savefig() here, using the well ID in the filename
        filename = f'well_{w}_fractions.png'
        filepath = os.path.join(subfolder, filename)
        plt.savefig(filepath, dpi=300)
        
    except np.AxisError:  # Add the necessary exception 
        print(f"Axis error for well {w}. Skipping this well.")
        continue
        


In [ ]:
#This is to enable the merge of both datasets on the well_ID
int_df = int_df.rename(columns={ "Destination Well": "well_ID"})
pooled_df['plate_well'] = pooled_df.index.astype(str)
pooled_df['well_ID'] = pooled_df['plate_well'].apply(lambda x: x.split('_')[1] if '_' in x else x)

In [ ]:
df = int_df.merge(
    pooled_df, 
    left_on='well_ID', 
    right_on='well_ID'
).apply(pd.to_numeric, errors='ignore')

df['conc_uM'] = 1e3 * df['pooled_integral'] / (df['pooled_vol'] * df['e280'] * df['path_length'])
df['conc_mgmL'] = 1e-3 * df['pooled_integral'] / (df['pooled_vol'] * df['OD280'] * df['path_length'])
df['pooled_mg'] = df['conc_mgmL'] * df['pooled_vol']
df['pooled_peak_estimated_MW'] = df['pooled_peak_vol'].apply(Vel2MW)
df['pooled_peak_agg_state'] = round(df['pooled_peak_estimated_MW'] / df['protomer_MW']).astype(int)
df['pooled_is_main'] = df['main_peak']==df['pooled_peak_vol']
df['Destination Well'] = df['well_ID'] # ADDED TO MAKE THE OT2 WORK

# Plot pooled statistics.
fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(10,6))
ax = ax.reshape(-1)

for i, (k, v) in enumerate({
        'pooled_vol':'Pooled volume / mL',
        'pooled_mg':'Pooled amount / mg',
        'conc_uM':'Concentration / $\mu$M',
        'conc_mgmL':'Concentration / mg mL$^{-1}$',
    }.items()):
    
    sns.histplot(df[k], ax=ax[i], element='step', bins=15, alpha=0.1, log_scale=False if 'vol' in k else True)
    sns.ecdfplot(df[k], ax=ax[i].twinx(), alpha=0.5, color='salmon')
    ax[i].spines['right'].set_visible(True)
    ax[i].set(xlabel=v)

plt.tight_layout()
plt.savefig('pooled_results.png', dpi=300)
plt.show()

In [ ]:
# Save DataFrame to CSV -- for more analysis
df.to_csv('pooling_df.csv', index=False)

## Generate OT-2 script

In [ ]:
find_opt_concentration(df)

In [ ]:
desired_conc = 20 # uM

df['buffer_vol'] = df.pooled_vol * (df.conc_uM - desired_conc) / desired_conc 
df['buffer_vol'] = df.apply(correct_buffer, args=(2,), axis=1)
df['normed_conc_uM'] = df.conc_uM * df.pooled_vol / (df.pooled_vol + df.buffer_vol)

frac_correct = np.isclose(df.normed_conc_uM.to_list(), desired_conc, atol=0.05).sum() / len(df)
print(f'{100 * frac_correct:.1f}% of wells will have the desired concentration.')

# Plot normalization statistics.
fig, ax = plt.subplots(ncols=2, figsize=(10,3.5))
ax = ax.reshape(-1)

for i, (k, v) in enumerate({
        'normed_conc_uM':'Normalised concentration / $\mu$M',
        'buffer_vol':'Volume buffer / mL',
    }.items()):
    
    sns.histplot(df[k], ax=ax[i], element='step', alpha=0.1, bins=30)
    sns.ecdfplot(df[k], ax=ax[i].twinx(), alpha=0.5, color='salmon')
    ax[i].spines['right'].set_visible(True)
    ax[i].set(xlabel=v)

plt.tight_layout()
plt.savefig('normalization_results.png', dpi=300)
plt.show()

In [ ]:
gen_ot2_script(
    df.dropna(),
    sec_instrument,
    normalize=True,
    template=SEC_POOL_TEMPLATE  # from SAPP_DMX
)

## Save data to disk

The following two files will be generated:
- `.csv` file containing only the scalar entries of the dataframe.
- `.h5` file containing everything, including arrays (i.e. SEC traces, etc...).

In [ ]:
today = datetime.datetime.today().strftime('%Y-%m-%d')

def is_scalar(column):
    return np.isscalar(column.iloc[0])  # Check the first element's type

# Save scalar columns to CSV
scalar_columns = [col for col in df.columns if is_scalar(df[col])]
df[scalar_columns].to_csv(f'{today}_expdata_df.csv')

# Selectively convert object columns to strings for HDF5 saving
# Assuming 'category' is a column meant to be categorical and safe to convert
columns_to_convert = ['category']  # Specify columns you know can be safely converted

for col in columns_to_convert:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = df[col].astype(str)

# Save entire DataFrame to HDF5
df.to_hdf(f'{today}_expdata_df.h5', key='df', complevel=9)

print(f"Scalar columns saved to {today}_expdata_df.csv")
print(f"Entire DataFrame saved to {today}_expdata_df.h5")